In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [6]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
11,model_10_6_0,1.000000,0.691078,1.000000,1.000000,1.000000,2.746018e-08,0.183389,6.654795e-14,1.500709e-08,7.503581e-09,0.000048,0.000166,1.000000,0.000173,172.821058,256.923490,"Hidden Size=[17], regularizer=0.5, learning_ra..."
24,model_27_4_16,1.000000,0.705708,1.000000,0.999999,1.000000,3.646659e-08,0.174704,8.040562e-08,1.113030e-07,9.585433e-08,0.000074,0.000191,1.000000,0.000199,204.253738,307.858184,"Hidden Size=[21], regularizer=0.3, learning_ra..."
25,model_27_4_15,1.000000,0.705719,1.000000,0.999999,1.000000,3.647289e-08,0.174698,7.649718e-08,1.066645e-07,9.156297e-08,0.000074,0.000191,1.000000,0.000199,204.253393,307.857838,"Hidden Size=[21], regularizer=0.3, learning_ra..."
26,model_27_4_17,1.000000,0.705698,1.000000,0.999999,1.000000,3.664964e-08,0.174710,8.405902e-08,1.156527e-07,9.985585e-08,0.000073,0.000191,1.000000,0.000200,204.243724,307.848170,"Hidden Size=[21], regularizer=0.3, learning_ra..."
27,model_27_4_14,1.000000,0.705731,1.000000,1.000000,1.000000,3.669286e-08,0.174691,7.227262e-08,1.017067e-07,8.698966e-08,0.000075,0.000192,1.000000,0.000200,204.241367,307.845813,"Hidden Size=[21], regularizer=0.3, learning_ra..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1200,model_3_9_10,0.999941,0.664030,0.999993,0.999995,0.999994,3.513233e-05,0.199446,6.937039e-06,2.689961e-06,4.813500e-06,0.001852,0.005927,1.000038,0.006180,142.512778,216.864203,"Hidden Size=[15], regularizer=0.3, learning_ra..."
1201,model_3_9_12,0.999941,0.664030,0.999993,0.999995,0.999994,3.513233e-05,0.199446,6.937039e-06,2.689961e-06,4.813500e-06,0.001852,0.005927,1.000038,0.006180,142.512778,216.864203,"Hidden Size=[15], regularizer=0.3, learning_ra..."
1202,model_3_9_23,0.999941,0.664030,0.999993,0.999995,0.999994,3.513233e-05,0.199446,6.937039e-06,2.689961e-06,4.813500e-06,0.001852,0.005927,1.000038,0.006180,142.512778,216.864203,"Hidden Size=[15], regularizer=0.3, learning_ra..."
1203,model_3_9_13,0.999941,0.664030,0.999993,0.999995,0.999994,3.513233e-05,0.199446,6.937039e-06,2.689961e-06,4.813500e-06,0.001852,0.005927,1.000038,0.006180,142.512778,216.864203,"Hidden Size=[15], regularizer=0.3, learning_ra..."
